# 🛠️ Notebook 2: Hotel Management — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/hotel-management
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from dataclasses import dataclass, field
from datetime import date
from enum import Enum
import itertools

class RoomType(Enum):
    SINGLE = 100
    DOUBLE = 150
    SUITE  = 300

@dataclass
class Room:
    number: int
    type: RoomType

@dataclass
class Guest:
    id: str
    name: str

@dataclass
class Reservation:
    id: int
    room: Room
    guest: Guest
    check_in: date
    check_out: date
    cancelled: bool = False
    @property
    def nights(self):
        return (self.check_out - self.check_in).days
    @property
    def total(self):
        return self.nights * self.room.type.value

def overlaps(a_in, a_out, b_in, b_out):
    return a_in < b_out and b_in < a_out

class Hotel:
    _rid = itertools.count(1)
    def __init__(self, rooms):
        self.rooms = {r.number: r for r in rooms}
        self.reservations = []

    def is_available(self, room_number, check_in, check_out):
        for r in self.reservations:
            if r.cancelled or r.room.number != room_number:
                continue
            if overlaps(check_in, check_out, r.check_in, r.check_out):
                return False
        return True

    def search(self, room_type, check_in, check_out):
        return [r for r in self.rooms.values()
                if r.type == room_type and self.is_available(r.number, check_in, check_out)]

    def reserve(self, room_number, guest, check_in, check_out):
        if check_out <= check_in:
            raise ValueError('check_out must be after check_in')
        if not self.is_available(room_number, check_in, check_out):
            raise RuntimeError(f'room {room_number} not available for those dates')
        r = Reservation(next(Hotel._rid), self.rooms[room_number], guest, check_in, check_out)
        self.reservations.append(r)
        return r

    def cancel(self, reservation):
        reservation.cancelled = True


## Walk-through

In [ ]:
hotel = Hotel([
    Room(101, RoomType.SINGLE),
    Room(102, RoomType.SINGLE),
    Room(201, RoomType.DOUBLE),
    Room(301, RoomType.SUITE),
])

ada  = Guest('G1','Ada')
grace= Guest('G2','Grace')

r1 = hotel.reserve(101, ada,  date(2026,5,1), date(2026,5,4))
print('reserved', r1.id, 'total =', r1.total)

# overlapping booking on same room should fail
try:
    hotel.reserve(101, grace, date(2026,5,3), date(2026,5,5))
except RuntimeError as e:
    print('expected:', e)

# non-overlapping dates on same room are fine
r2 = hotel.reserve(101, grace, date(2026,5,4), date(2026,5,6))
print('reserved', r2.id, 'nights =', r2.nights)

# search for any single available May 4-6
avail = hotel.search(RoomType.SINGLE, date(2026,5,4), date(2026,5,6))
print('available singles:', [r.number for r in avail])


### Key OOP ideas in play
- **Encapsulation**: the `Hotel` owns availability logic.
- **Value objects**: `Reservation` is immutable-ish; totals are computed.
- **Guard clauses** validate inputs.